# Meridian National Bank (MNB) — Credit Card Risk Analytics
## A Data Analyst's Workbook: From Raw Data to Business Insight

**How to use this notebook:**
- Cells marked `# TODO` are for you. Attempt them before looking at the companion DOCX workbook's answer keys.
- Cells marked `# 🔍 PREDICT FIRST` ask you to write your guess before running code — in a markdown cell, or on paper. Predicting before running is what builds intuition; skipping it turns this into copy-paste.
- Some code is intentionally broken. Your job is to diagnose it, not just run it.
- Place the `data/` folder (from the dataset download) in the same directory as this notebook.

**The scenario:** You are a data analyst at a third-party analytics vendor supporting Meridian National Bank (MNB), a fictional US regional bank. You've been handed several extracts from MNB's systems. Nobody has told you the data is clean — because it isn't.


In [ ]:
# Setup — run this first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sqlite3

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
sns.set_style('whitegrid')

DATA_DIR = 'data/'
print("Setup complete.")


---
# Phase 1 — Understand the Business Problem

Before writing a single line of pandas, a data analyst asks questions. This phase has almost no code — that's the point.

### The business ask (as MNB's VP of Credit Risk phrased it to you):

> "We want to understand which customer characteristics are associated with credit-card default, so underwriting can tighten policy in the right places."

### 🧠 Exercise 1.1 — Convert the vague ask into specifics

Answer these in your own words in the markdown cell below, before touching data:

1. What is the **unit of analysis** — what does one row represent for this question? (customer? account? account-month?)
2. What is the **target variable**?
3. What is the **target population**?
4. What **time period** is relevant, and why does that matter?
5. Who will **use** this analysis, and what decision will it inform?
6. What makes a finding **useful** vs merely "interesting"?


**Your answers:**

1. Unit of analysis: 
2. Target variable: 
3. Target population: 
4. Time period: 
5. Audience & decision: 
6. Useful vs interesting: 


### 📋 Answer Key 1.1

<details>
<summary>Click to expand</summary>

1. **Unit of analysis**: Most naturally the *account* — a customer can hold multiple cards, and default is recorded per account. But `monthly_performance` is account-*month* grain, so you must decide: analyze snapshots, or roll up to one row per account?
2. **Target variable**: `default_flag` in `accounts.csv` — a single flag, not time-varying. Is it "ever defaulted in the window" or "defaulted as of the latest snapshot"? Here, treat it as "defaulted at any point during the observation window," and document that assumption.
3. **Target population**: Probably all accounts opened before the observation window closed — but should already-closed accounts that defaulted before closing count? A judgment call to document, not guess silently.
4. **Time period**: `monthly_performance` spans Oct 2024–Sep 2025. If default is seasonal, your window choice changes your answer.
5. **Audience**: Underwriting policy team, considering approval threshold changes. This means findings need to be **actionable at application time** — using variables known *before* the account existed (age, income, credit score at application), not variables that only exist because the account already exists (utilization after years of use).
6. **Useful vs interesting**: "Customers named Robert default more" might be true and is not useful. "Customers with utilization above 90% at any point show a default rate several times higher than those under 30%" is useful and partly actionable — though still not proof of causation.

</details>


### ✍️ Exercise 1.2 — Bad question → Better question → Best question

For each bad question, write a Better and then a Best version using the **Business → Data → Analysis → Decision** framework (see DOCX Section 4 for the worked example).

| Bad | Better | Best |
|---|---|---|
| "Tell me about our risky customers." | *(your answer)* | *(your answer)* |
| "Why are defaults increasing?" | *(your answer)* | *(your answer)* |
| "Does income matter for default?" | *(your answer)* | *(your answer)* |

There's no single correct answer — check your Best version against the DOCX rubric: does it specify population, time period, and a measurable comparison?


---
# Phase 2 — Data Gathering

MNB's data lives in five different formats across two "systems." Real analysts constantly pull from mismatched sources. Let's gather each one and note what each format requires.

### 🔍 PREDICT FIRST
Before running the cells below: which of these five formats do you expect to load *without any extra arguments*, and which do you expect to need special handling (encoding, delimiter, sheet name, nested structure)? Write a one-line guess for each file.


In [ ]:
# 2.1 — CSV: the simplest case (usually)
customers = pd.read_csv(DATA_DIR + 'customers.csv')
customers.head()


In [ ]:
# 2.2 — TXT: pipe-delimited, not comma-delimited
# TODO: read_csv can read ANY delimiter via the `sep` argument. Try the default first and see what breaks.
branches = pd.read_csv(DATA_DIR + 'branch_lookup.txt')  # <-- this line is deliberately wrong. Fix it.
branches.head()


**What just happened?** `branch_lookup.txt` is pipe-delimited (`|`), not comma-delimited. Without `sep='|'`, pandas reads the entire row as a single column. This is one of the most common "gathering" bugs — the file extension `.txt` doesn't tell you the delimiter.

Fix the cell above, then continue.


In [ ]:
# 2.3 — Excel: multiple sheets
# TODO: read_excel() with sheet_name=None returns a dict of {sheet_name: DataFrame}. Try it.
apps_sheets = pd.read_excel(DATA_DIR + 'applications.xlsx', sheet_name=None)
print(apps_sheets.keys())
applications = apps_sheets['Applications']
decision_codes = apps_sheets['Decision_Codes']
branch_reference = apps_sheets['Branch_Reference']
applications.head()


In [ ]:
# 2.4 — JSON: nested structure
with open(DATA_DIR + 'bureau_data.json') as f:
    bureau_raw = json.load(f)

print(type(bureau_raw), len(bureau_raw))
print(bureau_raw[0])  # inspect one record — note the nested dict and list inside


### 🧠 Exercise 2.1 — Flattening nested JSON

`bureau_raw[0]` has a nested `public_records` dict and a `collections_accounts` list. If you `pd.DataFrame(bureau_raw)` directly, those nested structures stay as Python objects inside cells (not usable for analysis).

**Task:** Use `pd.json_normalize()` to flatten `bureau_raw` into a proper tabular DataFrame, with `public_records.bankruptcies` and `public_records.liens` as their own columns. Leave `collections_accounts` as-is for now (it's a list, and you'll handle it separately in Section 9 - Feature Engineering).


In [ ]:
# TODO: your code here
bureau = None  # replace


<details>
<summary>💡 Hint 1</summary>

Look at `pd.json_normalize()`'s docstring — it has a `sep` parameter for how to name flattened nested columns.
</details>

<details>
<summary>💡 Hint 2</summary>

`pd.json_normalize(bureau_raw)` alone will already flatten `public_records` automatically. Nested lists (`collections_accounts`) are NOT flattened by default — that's expected.
</details>

<details>
<summary>✅ Solution</summary>

```python
bureau = pd.json_normalize(bureau_raw)
bureau.columns.tolist()
```
Why this works: `json_normalize` walks nested dicts and creates `parent.child` column names by default. Lists are left as object columns because there's no single obvious way to flatten a variable-length list into columns — that's a modeling decision, not a mechanical one.
</details>


In [ ]:
# 2.5 — SQLite database: multiple related tables via SQL
conn = sqlite3.connect(DATA_DIR + 'mnb_portfolio.db')

# TODO: list the tables in this database
# Hint: sqlite_master is a special table that lists all tables/indexes
tables_query = """SELECT name FROM sqlite_master WHERE type='table';"""
pd.read_sql(tables_query, conn)


In [ ]:
# TODO: Write a SQL query that pulls customer_id, credit_score, and account_id, credit_limit
# by JOINing customers and accounts within the SQLite database itself (not with pandas .merge())
query = """
-- your SQL here
"""
# result = pd.read_sql(query, conn)
# result.head()


<details>
<summary>✅ Solution</summary>

```python
query = '''
SELECT c.customer_id, c.credit_score, a.account_id, a.credit_limit
FROM customers c
JOIN accounts a ON c.customer_id = a.customer_id
'''
result = pd.read_sql(query, conn)
```

**Why practice this in SQL AND pandas?** In real jobs, sometimes the join needs to happen at the database layer (too much data to pull everything into memory), and sometimes in pandas (more flexible transformations after). Knowing both means you're not stuck when one path is blocked.
</details>


In [ ]:
# 2.6 — The second CSV source (for later merge practice)
customers_dup_source = pd.read_csv(DATA_DIR + 'customers_dup_source.csv')
monthly_performance = pd.read_csv(DATA_DIR + 'monthly_performance.csv')
payments = pd.read_csv(DATA_DIR + 'payments.txt', sep='|')
accounts = pd.read_csv(DATA_DIR + 'accounts.csv')

print("All tables loaded:")
for name, df in [('customers',customers), ('accounts',accounts), ('monthly_performance',monthly_performance),
                  ('payments',payments), ('applications',applications), ('bureau',bureau),
                  ('branches',branches), ('customers_dup_source', customers_dup_source)]:
    print(f"  {name}: {df.shape}")


---
# Phase 3 — Data Assessment (Initial Investigation)

**Do not clean yet.** First, understand what you have. Every pandas command below answers a specific question — build the habit of thinking "what am I trying to find out?" before typing.

| Question | Pandas operation |
|---|---|
| How many rows and columns do I have? | `.shape` |
| What do the first/last few rows look like? | `.head()` / `.tail()` |
| What are the data types, and are there missing values? | `.info()` |
| What's the basic statistical shape of numeric columns? | `.describe()` |
| How many unique values does each column have? | `.nunique()` |
| Exactly how many values are missing, per column? | `.isna().sum()` |
| Are there duplicate rows? | `.duplicated().sum()` |


In [ ]:
# 3.1 — customers.csv assessment
customers.shape


In [ ]:
customers.info()


### 🧠 Exercise 3.1 — Interpret `.info()`

Looking at the `.info()` output above, answer without writing code:

1. Which column(s) have a `Non-Null Count` less than the total row count? What does that tell you?
2. `annual_income` — what dtype is it, and is that what you'd expect for a currency column? If not, why might that be?
3. `age` and `credit_score` are `int64` with **no missing values shown** — does that guarantee the values are all valid? (Think back to Phase 1: "missing" and "invalid" are not the same problem.)


### 📋 Answer Key 3.1

<details><summary>Click to expand</summary>

1. Columns with `email`, `phone`, `employment_status`, `annual_income` (among others) will show fewer non-null values than total rows — meaning explicit missing values (`NaN`) exist there.
2. `annual_income` is likely `object` dtype, not `float64`. That's your first clue that something is stored as text — possibly the `$1,234.56`-formatted values you'll find in Phase 4. `.info()` catching this BEFORE `.describe()` is exactly why order matters: `.describe()` on an object column gives you count/unique/top/freq instead of mean/std, silently hiding the numeric summary you wanted.
3. **No.** `.isna().sum()` only catches explicit `NaN`/`None`. A `credit_score` of `950` or an `age` of `-5` is a *valid-looking, non-null integer* that is nonetheless *impossible*. This is the difference between **completeness** (missing) and **validity** (invalid) — two separate dimensions of the Data Quality Framework in Phase 4.

</details>


In [ ]:
customers.describe()


In [ ]:
# TODO: run .describe() on the categorical/object columns only, using include='object'


<details><summary>💡 Hint</summary>

`customers.describe(include='object')` — the output shows `count`, `unique`, `top`, `freq` instead of numeric stats.
</details>

### 🧠 Exercise 3.2 — Spot the problem from `describe(include='object')`
Look at the `unique` count for `employment_status`. You created only 4 employment categories in principle (Employed, Self-Employed, Unemployed, Retired). Is `unique` close to 4? If not, what does that tell you, and which Phase 4 data-quality dimension does it fall under?


In [ ]:
customers.nunique()


In [ ]:
customers.isna().sum().sort_values(ascending=False)


In [ ]:
customers.duplicated().sum()


### 🧠 Exercise 3.3 — Why is `duplicated().sum()` probably wrong here

`.duplicated()` by default flags a row as a duplicate only if **every column** matches an earlier row exactly. Recall from the data description that `customers.csv` contains not just exact duplicates but also *near-duplicates* — same `customer_id`, slightly different `annual_income`/`email` formatting.

**Task:** Write code to count how many `customer_id` values appear more than once, regardless of whether the other columns match exactly.


In [ ]:
# TODO: your code here


<details><summary>✅ Solution</summary>

```python
customers['customer_id'].value_counts()
customers['customer_id'].duplicated().sum()  # count of customer_ids that are NOT the first occurrence
```

**Why this matters more than `.duplicated()` on the whole row:** `.duplicated()` (whole-row) undercounts, because near-duplicate rows with slightly different income values are NOT flagged — yet they represent the same real-world customer and will double-count that customer in any aggregate statistic (e.g. average income, default rate) unless you resolve them.
</details>


In [ ]:
# Repeat the assessment pattern on the other tables — build the habit
accounts.info()


In [ ]:
monthly_performance.info()


### ✍️ Exercise 3.4 — Assessment checklist, self-directed

Run the same six-question assessment pattern (`.shape`, `.info()`, `.describe()`, `.nunique()`, `.isna().sum()`, duplicate check) on `payments`, `applications`, and `bureau`. For each table, write down **in a markdown cell** at least two specific things you noticed that will need attention in the cleaning phase. Be specific ("payment_amount has 6 negative values" beats "some values look off").


In [ ]:
# Your assessment code for payments, applications, bureau here


---
# Phase 4 — The Data Quality Framework

A reusable lens for categorizing any data problem you find. See the DOCX workbook Section 7 for the full reference table with credit-risk examples for each dimension. Quick summary:

| Dimension | Question it answers |
|---|---|
| **Accuracy** | Does the value represent reality correctly? |
| **Completeness** | Is required information missing? |
| **Consistency** | Is the same thing represented the same way everywhere? |
| **Validity** | Does the value follow expected rules/format/range? |
| **Uniqueness** | Are records duplicated? |
| **Integrity** | Do relationships between tables hold (foreign keys valid)? |

### ✍️ Exercise 4.1 — Categorize what you found

Go back to your notes from Exercise 3.4 (and anything you noticed earlier in `customers`/`accounts`). For each issue, assign it to ONE of the six dimensions above. Some issues could arguably fit two dimensions — pick the primary one and be ready to justify it.

Example: `employment_status` having values `"Employed"`, `"employed"`, `"EMPLOYED"`, `" Employed "` → **Consistency** (same real-world category, different representations) — NOT Validity, because each individual string isn't inherently invalid text.


In [ ]:
# 🔍 PREDICT FIRST: before running, guess how many DISTINCT string variants of
# "Self-Employed" you think exist in the employment_status column. Write your guess.

customers['employment_status'].value_counts(dropna=False)


### 🧠 Exercise 4.2 — Investigate before you fix: the `credit_score = 950` case

The data-quality changelog (hidden from you until you check the DOCX answer key) says a handful of `credit_score` values fall outside the valid FICO range of 300–850, including some very unusual values like `0` and `1200`.

**Before writing any cleaning code, answer:**
1. Is `0` more likely a genuine data entry error, or a placeholder/null code that got merged into the numeric column?
2. Would you treat `0` and `1200` the same way? Why or why not?
3. What would you want to ask the source system owner before deciding how to handle these?

This is deliberately a reasoning exercise, not a coding one — real analysts spend more time on questions like these than on the fix itself.


In [ ]:
# Find the out-of-range credit scores
customers[(customers['credit_score'] < 300) | (customers['credit_score'] > 850)][['customer_id','credit_score']]


---
# Phase 5 — Data Cleaning

**The wrong mental model:** "Find dirty data, apply `.dropna()` or `.fillna()`, move on."

**The right mental model:** For every issue, ask:
- *Why* is this value missing/wrong?
- *How much* of my data does this affect?
- Is the missingness/error **random**, or does it correlate with something (like employment status)?
- Can I safely **drop** the row, or would that bias my analysis?
- Should I **impute** a value, and with what — mean, median, a model, or a business-rule default?
- Should I create a **missing indicator** column instead of (or in addition to) imputing?
- Could the missingness **itself carry business meaning**? (Hint: revisit who is missing `annual_income` in this dataset.)

## 5.1 Missing values


In [ ]:
# 🔍 PREDICT FIRST: You already saw employment_status and annual_income have missing values.
# Before running this cell, predict: do you think income is MORE or LESS likely to be missing
# for Unemployed/Retired customers vs Employed customers? Why might that be operationally true
# (think about how a bank collects this field)?

customers.groupby('employment_status', dropna=False)['annual_income'].apply(lambda s: s.isna().mean() if s.dtype != 'object' else np.nan)


That groupby will likely error or behave oddly — `annual_income` is still stored as a mix of numbers and `"$X,XXX.XX"` strings at this point (object dtype), so `.isna()` works but you can't safely reason about the *numeric* missingness rate until you've fixed the dtype. This is a preview of why **cleaning order matters**: fix data types before you can even reliably assess certain things.

### 🧠 Exercise 5.1 — Fix `annual_income`'s dtype first, then re-check missingness by employment status


In [ ]:
# TODO: Convert annual_income to a proper numeric column.
# It currently contains: real floats, NaN, AND strings like "$56,644.95"
# Hint: str.replace() to strip $ and commas, then pd.to_numeric() with errors='coerce'

# your code here


<details><summary>💡 Hint</summary>

```python
customers['annual_income'] = (
    customers['annual_income']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)
customers['annual_income'] = pd.to_numeric(customers['annual_income'], errors='coerce')
```
Watch out: `.astype(str)` will turn real `NaN` into the *string* `"nan"`, which `pd.to_numeric(errors='coerce')` will correctly turn back into `NaN` — but only because of `errors='coerce'`. Without it, this line would crash.
</details>

<details><summary>✅ Full solution + why it works</summary>

```python
customers['annual_income'] = (
    customers['annual_income']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace('nan', np.nan)  # undo the str→"nan" conversion for true NaNs
)
customers['annual_income'] = pd.to_numeric(customers['annual_income'], errors='coerce')
```

**Common wrong approach:** Using `.str.replace('[\$,]', '', regex=True)` without handling `NaN` first often works fine too — but doing `pd.to_numeric()` WITHOUT `errors='coerce'` will crash the moment it hits any row it can't parse, including legitimately malformed values you actually want to *catch*, not silently allow to crash your whole pipeline.
</details>


In [ ]:
# Now re-run the missingness-by-employment-status check
missingness_by_emp = customers.groupby('employment_status', dropna=False)['annual_income'].apply(lambda s: s.isna().mean())
missingness_by_emp.sort_values(ascending=False)


### 🧠 Exercise 5.2 — Decide: drop, impute, or flag?

Given what you just saw — income missingness is elevated for Unemployed/Retired customers, not random — answer:

1. If you simply `dropna()` rows missing `annual_income`, which customer segments get disproportionately removed from your dataset? What would that do to any later default-rate analysis by employment status?
2. If you impute missing income with the **overall mean**, what happens to Unemployed/Retired customers' income specifically?
3. Propose a better strategy. (There's more than one defensible answer — the DOCX answer key gives one.)


<details><summary>📋 One defensible answer</summary>

Dropping rows would disproportionately remove Unemployed/Retired customers (since they're missing income more often), which would bias any downstream analysis of default rate by employment status — you'd be studying a *non-representative* subset of exactly the segment most likely to be risky.

Imputing with the overall mean would artificially inflate the (already low) income of Unemployed/Retired customers, potentially making them look less risky than they are on an income basis — masking a real signal.

A more defensible approach: impute **within employment_status groups** (e.g., median income for Unemployed customers, computed only from Unemployed customers with known income) AND add a `income_was_missing` indicator column, so any model or analysis downstream can still use the fact that it was missing as a signal in its own right — since here, missingness itself is informative (Unemployed/Retired customers are less likely to have a reliable reported income on file).
</details>


In [ ]:
# TODO: implement group-wise median imputation + a missing-indicator column
customers['income_was_missing'] = customers['annual_income'].isna()

# your group-wise imputation code here


<details><summary>✅ Solution</summary>

```python
customers['income_was_missing'] = customers['annual_income'].isna()
customers['annual_income'] = customers.groupby('employment_status')['annual_income'].transform(
    lambda s: s.fillna(s.median())
)
```
**Why `.transform()` and not `.apply()` + manual reassignment:** `.transform()` returns a Series aligned to the ORIGINAL index/shape, so it can be assigned straight back to the column. `.apply()` on a groupby object without care can return a differently-shaped or differently-indexed result. This is a good moment to internalize: `.transform()` is for "same shape in, same shape out, computed per group."
</details>


## 5.2 Duplicates

In [ ]:
# 🔍 PREDICT FIRST: You already found customer_id duplicates in Exercise 3.3.
# Before running: do you expect the FIRST or the conflicting-value duplicate rows to have
# reliably "better" data? Is there any way to know which record is more recent/correct
# with the columns available?

dupe_ids = customers[customers['customer_id'].duplicated(keep=False)].sort_values('customer_id')
dupe_ids[['customer_id','first_name','annual_income','email']].head(20)


### 🧠 Exercise 5.3 — Deduplication strategy

Notice: nothing in this table tells you which duplicate record is "more recent" (no `last_updated` timestamp exists). This is realistic — you often don't get a clean tiebreaker.

**Task:** Decide and implement a deduplication rule. Options include: keep first, keep last, keep the row with the fewest missing values, or aggregate (e.g., average the conflicting numeric values). Document WHY you chose your rule in a markdown cell — there's no single right answer, but there IS a wrong non-answer ("I just used `.drop_duplicates()` and didn't think about it").


In [ ]:
# TODO: your deduplication code + a markdown cell explaining your reasoning


## 5.3 Invalid / impossible values

In [ ]:
# Age and credit_score impossible-value checks
bad_age = customers[(customers['age'] <= 0) | (customers['age'] > 100)]
bad_score = customers[(customers['credit_score'] < 300) | (customers['credit_score'] > 850)]
print("Impossible ages:", len(bad_age))
print("Impossible credit scores:", len(bad_score))
bad_age[['customer_id','age']]


### 🧠 Exercise 5.4 — Debugging exercise: broken cleaning code

The cell below is meant to cap `age` at a maximum plausible value of 100 and floor it at 18 (MNB doesn't issue cards to minors), but it has a bug. Find it and fix it.


In [ ]:
# BROKEN — find and fix the bug
def clean_age(age):
    if age > 100:
        age = 100
    if age < 18:
        age = 18
    return age

customers['age'] = customers['age'].apply(clean_age)
print(customers['age'].describe())


<details><summary>💡 Hint</summary>

Run `customers['age'].describe()` BEFORE the fix and look at the minimum. What was the original minimum value, and does clamping it to 18 make sense for values like `-5` or `0`? Is "floor at 18" even the right call for a negative number, or is `-5` actually a different KIND of problem (a sign error / data entry error) than "a legitimate 16-year-old applied"?
</details>

<details><summary>✅ Discussion</summary>

The code isn't syntactically broken — it runs. The BUG is conceptual: silently clamping `-5` to `18` treats a data entry error as if it were a real (if underage) customer, which is misleading. A `-5` is almost certainly a sign error or corrupted field, not "someone aged negative-five requested a card." The more defensible fix is to treat implausible values (negative, zero, or absurdly high like 210) as **invalid → convert to NaN → then decide on imputation**, rather than clamping them into a plausible-looking but fabricated range.

```python
customers['age'] = customers['age'].where((customers['age'] >= 18) & (customers['age'] <= 100))
# Now age has NaN where it was invalid — proceed to Section 5.1's missing-value decision process
```
This distinction — **clamping vs. nulling-then-deciding** — is a common point of failure for beginners, and a common interview question ("would you cap or null an outlier/invalid value, and why does it matter?").
</details>


## 5.4 Standardizing categories

In [ ]:
# 🔍 PREDICT FIRST: how many .replace() or .str operations do you think you need
# to collapse ALL the Self-Employed spelling variants into one canonical value?

customers['employment_status'].value_counts(dropna=False)


In [ ]:
# TODO: standardize employment_status
# 1. Strip whitespace
# 2. Normalize case
# 3. Map all variants (Self Employed, self-employed, SE, Self_Employed, etc.) to one canonical "Self-Employed"

# your code here


<details><summary>✅ Solution</summary>

```python
customers['employment_status'] = customers['employment_status'].str.strip().str.lower()

mapping = {
    'employed': 'Employed',
    'self-employed': 'Self-Employed',
    'self employed': 'Self-Employed',
    'self_employed': 'Self-Employed',
    'se': 'Self-Employed',
    'unemployed': 'Unemployed',
    'retired': 'Retired',
}
customers['employment_status'] = customers['employment_status'].map(mapping)
customers['employment_status'].value_counts(dropna=False)
```

**Why map() over a chain of .replace()?** A dictionary `.map()` is a single, readable, exhaustive lookup table — easy to audit ("did I cover every variant?") by comparing keys against `.value_counts()` BEFORE mapping. A long chain of `.str.replace()` calls can silently miss a variant or accidentally match a substring of something else.

**Trap:** `.map()` turns any value NOT in the dictionary into `NaN` — including any variant you forgot to list! Always check `.value_counts(dropna=False)` immediately after, comparing your before/after unique counts.
</details>


## 5.5 Invalid dates

In [ ]:
# date_of_birth has some genuinely invalid entries: future dates, invalid months, unparseable text
customers['date_of_birth_parsed'] = pd.to_datetime(customers['date_of_birth'], errors='coerce')
customers[customers['date_of_birth_parsed'].isna()][['customer_id','date_of_birth']]


### 🧠 Exercise 5.5 — Compare and contrast: `errors='coerce'` vs `errors='raise'` vs `errors='ignore'`

Without running code, predict what each of these three would do differently on the `date_of_birth` column, given that some values are legitimately unparseable text like `"not_a_date"`:

- `pd.to_datetime(customers['date_of_birth'], errors='raise')`
- `pd.to_datetime(customers['date_of_birth'], errors='coerce')`
- `pd.to_datetime(customers['date_of_birth'], errors='ignore')` *(note: deprecated/removed in newer pandas — check your version)*

Which would you use during initial ASSESSMENT (Phase 3) vs during final CLEANING (Phase 5), and why might you want different behavior at each stage?


In [ ]:
# Also check for dates that parse fine but are logically impossible: future customer_since dates
customers['customer_since_parsed'] = pd.to_datetime(customers['customer_since'], errors='coerce')
future_signups = customers[customers['customer_since_parsed'] > pd.Timestamp.today()]
future_signups[['customer_id','customer_since']]


## 5.6 Referential integrity

In [ ]:
# 🧠 Exercise 5.6 — Find orphan foreign keys
# Some accounts.csv rows reference a customer_id that does NOT exist in customers.csv.
# TODO: find them using a set difference or an anti-join (merge with indicator=True)

# your code here


<details><summary>💡 Hint</summary>

`pd.merge(accounts, customers[['customer_id']], on='customer_id', how='left', indicator=True)` — rows where `_merge == 'left_only'` exist in `accounts` but not `customers`.
</details>

<details><summary>✅ Solution</summary>

```python
check = accounts.merge(customers[['customer_id']], on='customer_id', how='left', indicator=True)
orphans = check[check['_merge'] == 'left_only']
print(f"{len(orphans)} accounts reference a customer_id not present in customers.csv")
orphans[['account_id','customer_id']]
```

**Business question this raises, not just a technical one:** should these orphan accounts be dropped from analysis, or does their existence suggest a broader ETL/systems problem worth flagging to the data engineering team? A good analyst reports the *pattern*, not just silently drops the rows.
</details>


### ✍️ Exercise 5.7 — Full cleaning pass on `accounts.csv`

Apply the same reasoning process to `accounts.csv`:
- `credit_limit` has some zero/negative values — decide how to handle.
- `utilization_ratio` has a few negative values — decide how to handle (hint: negative utilization has no real-world meaning, unlike utilization slightly above 1.0 which CAN happen from fees/interest pushing balance over the limit).
- `account_status` has missing values.
- `card_type` has inconsistent capitalization.
- There are duplicate `account_id` rows.

Write your cleaning code below. Aim to produce a clean `accounts_clean` DataFrame you'll use for the rest of the notebook.


In [ ]:
# Your accounts.csv cleaning code here
accounts_clean = accounts.copy()

# TODO


### 🎯 Checkpoint

By this point you should have `customers` (cleaned) and `accounts_clean`. Before moving to Phase 6, verify:
```python
assert customers['customer_id'].duplicated().sum() == 0  # if you deduplicated
assert customers['annual_income'].dtype in ('float64','int64')
assert accounts_clean['credit_limit'].min() > 0
```
If any assertion fails, go back — Phase 6 (Wrangling) and Phase 7 (Feature Engineering) both assume clean inputs, and bugs here will silently corrupt everything downstream.


---
# Phase 6 — Data Wrangling

Cleaning fixes what's *wrong*. Wrangling reshapes what's *right* into the shape you need for analysis. This section builds the habit of choosing the correct operation for the question you're asking.

### Compare and contrast: filtering vs grouping vs aggregating vs merging vs reshaping

| Operation | What it does | Credit-risk example |
|---|---|---|
| **Filter** (`query`, boolean indexing) | Keep a subset of ROWS | "Only active accounts" |
| **Group + Aggregate** (`groupby().agg()`) | Collapse many rows into one summary row per group | "Average utilization per credit-score band" |
| **Transform** (`groupby().transform()`) | Compute per-group stat, but keep original row count | "Each row's deviation from its group's average" |
| **Merge** (`merge`) | Combine columns from two tables using a key | "Add customer credit_score onto each account row" |
| **Reshape** (`pivot_table`, `melt`) | Change from long to wide format or vice versa | "One row per customer, one column per month's balance" |


In [ ]:
# 6.1 loc vs iloc
# 🔍 PREDICT FIRST: what's the difference in output between these two lines?
print(customers.loc[0:3])    # label-based
print(customers.iloc[0:3])   # position-based


### 🧠 Exercise 6.1 — loc vs iloc, the trap

After you deduplicated and possibly filtered `customers` earlier, the index is no longer a clean `0, 1, 2, ...` sequence. `.loc[0:3]` selects by LABEL (inclusive of both ends) and will behave unpredictably if row 0, 1, 2, 3 don't all exist anymore. `.iloc[0:3]` always selects by POSITION (exclusive of the end, like standard Python slicing) regardless of what the labels are.

**Task:** Reset the index of your cleaned `customers` DataFrame, then explain in one sentence why `.iloc` is generally safer for "give me the first N rows" after any filtering/dropping operation.


In [ ]:
customers = customers.reset_index(drop=True)


In [ ]:
# 6.2 Filtering with query() vs boolean indexing — same result, different readability
high_util_boolean = accounts_clean[accounts_clean['utilization_ratio'] > 0.8]
high_util_query = accounts_clean.query('utilization_ratio > 0.8')
assert len(high_util_boolean) == len(high_util_query)
print("Both approaches agree:", len(high_util_boolean), "rows")


In [ ]:
# 6.3 assign() for chainable column creation
# TODO: use .assign() to add a 'high_utilization' boolean column (>0.8) WITHOUT overwriting accounts_clean directly
# This is useful when you want to preview a transformation before committing to it

preview = accounts_clean.assign(high_utilization = lambda df: df['utilization_ratio'] > 0.8)
preview[['account_id','utilization_ratio','high_utilization']].head()


### 🧠 Exercise 6.2 — merge() types

`accounts_clean` has a `customer_id` foreign key. `customers` has the primary key. You want ONE row per account, enriched with that customer's `credit_score`, `age`, and `annual_income`.

1. Which merge type (`left`, `right`, `inner`, `outer`) keeps ALL accounts, even the orphan ones you found in Exercise 5.6?
2. Which merge type would SILENTLY DROP the orphan accounts, potentially hiding a data quality problem from anyone looking at the merged result?
3. Run both and compare row counts to confirm your prediction.


In [ ]:
# TODO: two merges — one that keeps orphans, one that drops them. Compare row counts.
merged_left = None   # your code
merged_inner = None  # your code

# print(len(accounts_clean), len(merged_left), len(merged_inner))


<details><summary>✅ Solution</summary>

```python
merged_left = accounts_clean.merge(customers, on='customer_id', how='left')
merged_inner = accounts_clean.merge(customers, on='customer_id', how='inner')
print(len(accounts_clean), len(merged_left), len(merged_inner))
```

`how='left'` keeps every row from `accounts_clean` (the left table), filling customer columns with `NaN` for orphans. `how='inner'` only keeps rows where the key matches in BOTH tables — orphans vanish silently. This is exactly why Exercise 5.6 mattered: if you inner-joined without first *knowing* about the orphans, you'd never notice they disappeared, and any portfolio-level "total accounts" count downstream would quietly be wrong.
</details>


In [ ]:
# Build the account-level analysis table you'll use for the rest of this notebook
account_analysis = accounts_clean.merge(customers, on='customer_id', how='left')
account_analysis.shape


### 🧠 Exercise 6.3 — groupby().agg() vs pivot_table(): same question, two tools

**Question:** What is the average utilization ratio and average credit_score, broken out by card_type?

Compute this TWO ways: once with `.groupby().agg()`, once with `.pivot_table()`. Compare the outputs — they should carry the same information in different shapes.


In [ ]:
# Method 1: groupby + agg
agg_result = account_analysis.groupby('card_type').agg(
    avg_utilization=('utilization_ratio','mean'),
    avg_credit_score=('credit_score','mean'),
    n_accounts=('account_id','count')
)
agg_result


In [ ]:
# TODO: Method 2 - same info using pivot_table()
pivot_result = None  # your code


<details><summary>✅ Solution</summary>

```python
pivot_result = account_analysis.pivot_table(
    values=['utilization_ratio','credit_score'],
    index='card_type',
    aggfunc='mean'
)
```
**When to reach for which:** `groupby().agg()` is more flexible — different aggregation functions per column, named outputs, and it composes cleanly into method chains. `pivot_table()` shines when you want a spreadsheet-style cross-tab, especially with TWO grouping dimensions (e.g., card_type as rows, account_status as columns) — that's awkward with plain `groupby()`.
</details>


In [ ]:
# 6.4 melt() — wide to long. Useful for monthly_performance-style time series.
# TODO: pivot monthly_performance to WIDE format (one row per account, one column per month's balance),
# then melt() it back to confirm you can reverse the operation.

wide = monthly_performance.pivot_table(values='balance', index='account_id', columns='statement_month')
wide.head()


In [ ]:
# TODO: melt 'wide' back into long format and compare to the original monthly_performance shape
# your code here


<details><summary>✅ Solution</summary>

```python
long_again = wide.reset_index().melt(id_vars='account_id', var_name='statement_month', value_name='balance')
```
`melt()` is the inverse of `pivot_table()`. You'll use `pivot_table` for READING/reporting (a human wants a cross-tab) and `melt` for STORAGE/analysis-readiness (most plotting and modeling functions expect long/tidy format: one row per observation).
</details>


---
# Phase 7 — Feature Engineering

**Compare and contrast — where does each activity belong?**

| Activity | Question it answers | Example |
|---|---|---|
| **Cleaning** | "Is this value correct/valid?" | Fixing `credit_score=950` → NaN or investigate |
| **Wrangling** | "Is this data in the right SHAPE?" | Merging accounts + customers into one table |
| **Feature Engineering** | "What NEW variable would better capture a business concept?" | `balance / credit_limit` → utilization |

Feature engineering always follows the pattern: **Raw variable(s) → transformation → new feature → business meaning.**

### 🧠 Exercise 7.1 — Predict before computing: utilization

`accounts_clean` already HAS a `utilization_ratio` column (it was generated for you as part of realistic account data). But imagine it didn't. You have `balance` and `credit_limit`.

Before running the cell: what formula would you use, and what value would you expect for a maxed-out card? What about a card with a fee/interest charge pushing it slightly over the limit?


In [ ]:
# Verify: recompute utilization from balance and credit_limit, and compare to the existing column
account_analysis['utilization_check'] = account_analysis['balance'] / account_analysis['credit_limit']
(account_analysis['utilization_check'] - account_analysis['utilization_ratio']).abs().max()


### ✍️ Exercise 7.2 — Build a full feature set

Using `account_analysis`, engineer the following features. For EACH one, write a one-line comment stating the business meaning — not just what the code does.

1. `age_band` — bucket `age` into `['18-25','26-35','36-45','46-55','56-65','65+']`
2. `credit_score_band` — bucket `credit_score` using standard-ish bands: `<580 Poor, 580-669 Fair, 670-739 Good, 740-799 Very Good, 800+ Exceptional`
3. `income_band` — your choice of reasonable bins (document your reasoning for the cutoffs)
4. `high_delinquency_flag` — `delinquency_count >= 3`
5. `debt_to_income_proxy` — `(balance * 12) / annual_income` — a rough proxy since we don't have full monthly-debt data; note the limitation of this proxy in a comment


In [ ]:
# TODO: implement all five features
# Hint: pd.cut() is the tool for band/bucket features

account_analysis['age_band'] = pd.cut(
    account_analysis['age'],
    bins=[17,25,35,45,55,65,120],
    labels=['18-25','26-35','36-45','46-55','56-65','65+']
)

# TODO: credit_score_band, income_band, high_delinquency_flag, debt_to_income_proxy


<details><summary>✅ Solution</summary>

```python
account_analysis['credit_score_band'] = pd.cut(
    account_analysis['credit_score'],
    bins=[299,579,669,739,799,850],
    labels=['Poor','Fair','Good','Very Good','Exceptional']
)

account_analysis['income_band'] = pd.cut(
    account_analysis['annual_income'],
    bins=[0,25000,50000,75000,100000,150000,np.inf],
    labels=['<25k','25-50k','50-75k','75-100k','100-150k','150k+']
)

account_analysis['high_delinquency_flag'] = account_analysis['delinquency_count'] >= 3

# NOTE: this is a proxy, not a true DTI — we lack full monthly debt obligations (mortgage, auto loan, etc.)
# It only captures credit-card debt annualized against income, so it UNDERSTATES true DTI for anyone
# with other debts. Document this limitation wherever the feature is used downstream.
account_analysis['debt_to_income_proxy'] = (account_analysis['balance'] * 12) / account_analysis['annual_income']
```

**Why `pd.cut()` and not a chain of `if/elif`?** `pd.cut()` is vectorized (fast on large data), self-documenting (bins and labels are visible in one place), and returns a proper `Categorical` dtype that sorts correctly in later groupbys/plots — an `if/elif` chain via `.apply()` works but is slower and easier to get subtly wrong (off-by-one on a boundary).
</details>


### 🧠 Exercise 7.3 — Interview-style question

*"Why not just use raw `credit_score` as a continuous variable instead of banding it into `credit_score_band`?"*

Think about this before checking the answer — there are legitimate arguments on both sides.


<details><summary>📋 Discussion</summary>

**Arguments for banding:**
- Matches how underwriting POLICY actually works in practice — cutoffs and tiers are how credit decisions get made and communicated to a business audience.
- Makes relationships easier to visualize and explain to non-technical stakeholders (a bar chart of default rate by band beats a scatter plot for a VP presentation).
- Can capture non-linear relationships without needing a more complex model.

**Arguments against (for continuous):**
- Banding throws away information — two customers at 671 and 739 are both "Good" but are meaningfully different.
- Bin boundaries are somewhat arbitrary and can create artificial cliffs in analysis.
- For statistical/ML modeling (not just descriptive reporting), continuous variables usually perform better.

**Best answer:** "It depends on the purpose — for EDA and stakeholder communication, banding aids interpretation. For a scoring MODEL, you'd typically keep the continuous variable (or use WOE-transformed bands, which you may already know from your SAS scorecard work) rather than lose information via simple integer bins." This shows you understand the difference between descriptive/exploratory use and predictive-modeling use.
</details>


---
# Phase 8 — Descriptive Statistics (with meaning, not just formulas)

For every statistic below, the real skill isn't computing it — pandas does that in one call. The skill is **reasoning about what it implies for a credit-risk dataset.**

### 🧠 Exercise 8.1 — Mean vs median: predict before computing

**Predict:** For `utilization_ratio`, do you expect the MEAN to be higher or lower than the MEDIAN? Why? (Think about the shape of the distribution — are there more customers clustered at low utilization with a long tail of high-utilization customers, or the reverse?)


In [ ]:
print("Mean utilization:", account_analysis['utilization_ratio'].mean().round(4))
print("Median utilization:", account_analysis['utilization_ratio'].median().round(4))


### 🧠 Exercise 8.2 — Interpret the gap

If mean > median (right-skewed distribution): what does that suggest about the shape of MNB's customer utilization patterns? Write your interpretation, then create a histogram to visually confirm it.


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
account_analysis['utilization_ratio'].hist(bins=40, ax=ax)
ax.set_xlabel('Utilization Ratio')
ax.set_title('Distribution of Credit Utilization')
plt.show()


### Reference: what each statistic tells you about credit-risk data

| Statistic | What it tells you | Credit-risk caution |
|---|---|---|
| **Mean** | The balance point / center of mass | Sensitive to outliers — one customer with 500% utilization (fees pushed way over) can drag the mean up |
| **Median** | The "typical" customer's value | More robust to outliers — often more representative for skewed financial data |
| **Mode** | The most common single value | Useful for categorical risk factors (most common `employment_status` among defaulters) |
| **Std Dev** | How spread out values are | High std dev in `credit_score` within a "band" would suggest your bands aren't well-differentiated |
| **Percentiles/Quartiles** | Where a value ranks relative to the population | "This customer's utilization is in the 95th percentile" is often more useful to underwriting than the raw number |
| **IQR** | The spread of the "middle" 50%, ignoring extremes | A standard, robust way to define outlier thresholds |
| **Skewness** | Direction and degree of asymmetry | Financial variables (income, balance) are almost always right-skewed — informs whether to log-transform before certain analyses |


In [ ]:
account_analysis[['annual_income','balance','utilization_ratio','credit_score']].describe()


### 🧠 Exercise 8.3 — Skewness and its practical consequence

Compute `.skew()` for `annual_income`. A skewness near 0 is roughly symmetric; positive skew means a long right tail (a few very high earners pulling the mean up).

**Follow-up question (interview-style):** If you were about to compute a Pearson correlation between `annual_income` and `default_flag`, would the skew in income concern you? What would you consider doing about it, and why?


In [ ]:
account_analysis['annual_income'].skew()


<details><summary>📋 Discussion</summary>

Highly skewed variables can distort correlation and some visualizations — a handful of extreme high-income outliers can dominate a scatter plot or inflate/deflate a correlation coefficient. A common practical step is a log transform (`np.log1p(income)`) before certain analyses, or using a rank-based correlation (Spearman) instead of Pearson, which is robust to the exact shape of the distribution. Whether you SHOULD do this depends on the specific analysis — always state your reasoning rather than transforming reflexively.
</details>


---
# Phase 9 — Exploratory Data Analysis (EDA)

**EDA is not "make some charts." EDA is a disciplined process:**

`Question → variable selection → visualization/statistic → interpretation`

Every chart in this section starts with a written question. If you can't state the question before the chart, you're not doing EDA — you're doing decoration.

## 9.1 Univariate analysis


In [ ]:
# Question: "What fraction of accounts in our portfolio have defaulted?"
default_rate = account_analysis['default_flag'].mean()
print(f"Overall default rate: {default_rate:.2%}")


### 🧠 Exercise 9.1 — Categorical univariate: employment_status

**Question to answer:** "What does the employment mix of MNB's card-holding customers look like?"

Produce a bar chart of `employment_status` value counts (or proportions). Then answer: does anything in this distribution surprise you, or does it match what you'd expect from a general-population bank customer base?


In [ ]:
# TODO: your bar chart here


<details><summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(6,4))
account_analysis['employment_status'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Customers by Employment Status')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.show()
```
</details>


## 9.2 Bivariate analysis — the heart of a default-driver investigation

In [ ]:
# 🔍 PREDICT FIRST: before running, sketch (on paper or in words) what you expect this
# chart to look like. Does default rate rise, fall, or stay flat as credit score band improves?

default_by_score = account_analysis.groupby('credit_score_band', observed=True)['default_flag'].mean()
fig, ax = plt.subplots(figsize=(8,4))
default_by_score.plot(kind='bar', ax=ax, color='#4C72B0')
ax.set_ylabel('Default Rate')
ax.set_title('Default Rate by Credit Score Band')
plt.xticks(rotation=0)
plt.show()
default_by_score


### 🧠 Exercise 9.2 — Interpret, then write a Finding/Evidence/Interpretation statement

Using the Phase 11 conclusion structure (introduced below), write:
- **Finding:**
- **Evidence:**
- **Interpretation:**
- **Business implication:**
- **Limitation:**

for the credit-score-band-vs-default-rate pattern you just observed.


In [ ]:
# TODO: utilization vs default — same pattern
# Question: "Do customers who use more of their available credit default more often?"

# your code here (hint: bucket utilization_ratio first, similar to credit_score_band)


In [ ]:
# TODO: income vs default
# Question: "Is there a relationship between income band and default rate?"

# your code here


### 🧠 Exercise 9.3 — Compare-and-contrast: correlation vs causation

You've now seen three variables (credit score, utilization, income) each show SOME association with default rate.

Write one paragraph explaining why "high utilization causes default" is an overreach, and what a more defensible statement would sound like. (Revisit Phase 11 for the exact language pattern to use.)


## 9.3 Multivariate — does the pattern hold across segments?

In [ ]:
# Question: "Is the utilization-default relationship consistent across income groups,
# or does income change the picture?" (This tests whether utilization is a real independent
# signal, or just a proxy for income that would disappear once you control for it.)

account_analysis['utilization_band'] = pd.cut(
    account_analysis['utilization_ratio'],
    bins=[-0.01, 0.3, 0.6, 0.9, np.inf],
    labels=['0-30%','30-60%','60-90%','90%+']
)

pivot = account_analysis.pivot_table(
    values='default_flag', index='utilization_band', columns='income_band', aggfunc='mean', observed=True
)
fig, ax = plt.subplots(figsize=(10,5))
sns.heatmap(pivot, annot=True, fmt='.1%', cmap='Reds', ax=ax)
ax.set_title('Default Rate by Utilization Band x Income Band')
plt.show()


### 🧠 Exercise 9.4 — Read the heatmap like an analyst, not a viewer

1. Within the SAME income band, does default rate still rise with utilization? (If yes, utilization carries independent signal beyond income.)
2. Are there any income-band/utilization-band cells with very few underlying customers, where the displayed rate might be unstable/unreliable? How would you check this? (Hint: you'd want a COUNT heatmap alongside the RATE heatmap.)


In [ ]:
# TODO: build the companion COUNT heatmap to check cell reliability
count_pivot = None  # your code

# your heatmap code


<details><summary>✅ Solution</summary>

```python
count_pivot = account_analysis.pivot_table(
    values='default_flag', index='utilization_band', columns='income_band', aggfunc='count', observed=True
)
fig, ax = plt.subplots(figsize=(10,5))
sns.heatmap(count_pivot, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Sample Size by Utilization Band x Income Band')
plt.show()
```
**Why this matters:** A cell showing "50% default rate" built from only 4 customers is not the same finding as "50% default rate" built from 400 customers. Reporting a rate without its underlying N is one of the most common ways analyses mislead — accidentally or otherwise.
</details>


### ✍️ Exercise 9.5 — Self-directed EDA

Pick TWO more variables from `account_analysis` (or engineer a new feature first) and repeat the `Question → variable selection → visualization → interpretation` process independently. Candidates: `age_band`, `delinquency_count`, `card_type`, `employment_status`. Write your own question first — don't skip straight to a chart.


In [ ]:
# Your self-directed EDA here


---
# Phase 10 — Are the Patterns Real, or Are They Data-Quality Artifacts?

A pattern found during EDA can be a genuine business signal OR an artifact of how the data was collected/loaded. Before trusting any EDA finding, sanity-check it against what you know from Phases 3-5.

### 🧠 Exercise 10.1 — The March 2025 trap

Recall from Phase 5 that `monthly_performance` has a batch of rows with negative `minimum_payment_due`, concentrated in March 2025 (a "systematic load issue," per your assessment). If you ran a time-series chart of average `minimum_payment_due` by month WITHOUT first cleaning this, what would you expect to see in March 2025, and would a viewer mistake this for a genuine business event (e.g., "the bank changed its minimum payment policy in March") rather than a data bug?


In [ ]:
# Demonstrate the trap: plot average minimum_payment_due by month BEFORE fixing the March issue
monthly_avg_dirty = monthly_performance.groupby('statement_month')['minimum_payment_due'].mean()
fig, ax = plt.subplots(figsize=(10,4))
monthly_avg_dirty.plot(kind='line', marker='o', ax=ax)
ax.set_title('Average Minimum Payment Due by Month (UNCLEANED — do not trust yet)')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# TODO: fix the March 2025 negative minimum_payment_due issue, then re-plot and compare
# your code here


**The lesson:** Every anomaly you see in a chart deserves the question "is this a real pattern, or something I already logged in my data-quality assessment?" before it goes into a business conclusion.


---
# Phase 11 — From Analysis to Conclusions

Every mini-project and finding should be written up using this structure:

> **Finding** → **Evidence** → **Interpretation** → **Business Implication** → **Limitation**

### The trap to avoid

❌ *"High utilization causes default."* — overclaims causation from observational, non-experimental data.

✅ *"Customers with utilization above 90% showed a default rate roughly 5x that of customers under 30% utilization in this dataset. This is an association, not a demonstrated causal effect — customers who are already in financial distress for OTHER reasons may both run up their utilization AND default, meaning utilization could be a symptom rather than a cause."*

### 🧠 Exercise 11.1 — Write two full conclusion statements

Using the structure above, write out full Finding/Evidence/Interpretation/Business Implication/Limitation statements for:
1. The credit-score-band vs default-rate relationship (Exercise 9.2 — you may already have a draft)
2. The utilization x income-band heatmap finding (Exercise 9.4)

Be deliberately skeptical of your own findings in the Limitation section — this is a skill, not a formality.


---
# Mini Projects

The exercises above walked you through each concept step by step. These projects remove the scaffolding — you decide the sequence.


## Project 1 — Customer Data Quality Audit

**Deliverable:** A short written audit (markdown cells below) of `customers.csv`, covering:
- Every data quality issue you find, categorized by the Phase 4 framework dimension (Accuracy / Completeness / Consistency / Validity / Uniqueness / Integrity)
- For each: how many rows affected, how you detected it, and your proposed fix
- A final cleaned `customers` DataFrame

You've already done pieces of this in Phases 3-5. This project asks you to consolidate it into one coherent, presentable audit — the way you'd hand it to a manager.


In [ ]:
# Project 1 workspace


## Project 2 — Credit Card Portfolio EDA

**Deliverable:** A univariate + bivariate exploration of the overall MNB card portfolio (not specifically default-focused) covering demographics, credit scores, utilization, balances, and income. At least 5 charts, each with a stated question and a written interpretation.


In [ ]:
# Project 2 workspace


## Project 3 — Multi-Source Risk Analysis

**Deliverable:** Combine `customers.csv`, `accounts.csv`, `bureau_data.json`, `applications.xlsx`, and `customers_dup_source.csv` into ONE analysis-ready DataFrame at the account grain, enriched with bureau data (inquiries, trade lines) where available.

**Specific challenges to handle:**
- `customers_dup_source.csv` has customers NOT present in `customers.csv`, and conflicting income/score values for customers that ARE present in both. Decide how to reconcile — do you prefer one source, average them, or flag the conflict?
- Not every customer has a bureau record (~93% coverage, by design) — decide how you'll handle the resulting missingness in bureau-derived columns after the merge.
- `applications.xlsx`'s `Applications` sheet has records where `customer_id` is null (a new applicant, not yet an existing customer) — these can't be merged onto the customer table and need a different handling strategy (exclude from this analysis, or analyze separately).


In [ ]:
# Project 3 workspace


## Project 4 — Default Risk EDA (End-to-End)

**Deliverable:** Starting from your Project 3 merged dataset, perform the FULL workflow: cleaning gaps you may have deferred, feature engineering (at least 3 new features beyond what you built in Phase 7), univariate + bivariate + at least one multivariate analysis, descriptive statistics, a customer segmentation (e.g., cross credit-score band with another dimension), and a written conclusions section using the Finding/Evidence/Interpretation/Business-Implication/Limitation structure for at least 3 distinct findings.


In [ ]:
# Project 4 workspace


## Project 5 — Analyst Case Study (No Roadmap Provided)

**You get only this:**

> *MNB's VP of Underwriting Policy has noticed that declined applications have been rising as a share of total applications over the past several months. She wants to know: is this a deliberate tightening of policy, a shift in the applicant pool, a data quality issue, or something else — and what should the underwriting team investigate next?*

You have access to `applications.xlsx` (including the `Decision_Codes` reference sheet), plus everything else in the `data/` folder if you find it relevant.

**No roadmap is given here on purpose.** Decide for yourself:
- What analytical questions does this business question break into?
- What data do you need, and is it sufficient, or are there gaps you'd need to flag?
- What would you assess, clean, and engineer?
- What EDA would you perform?
- What can you conclude, and what are the limitations?

Write your full analysis below. When done, check the DOCX workbook's Project 5 rubric (Section 18) — it will NOT tell you "the right answer" (there isn't one single one), but it will give you a checklist of the reasoning steps a strong analyst response should include, so you can self-assess.


In [ ]:
# Project 5 workspace — your independent analysis


---
# The Analyst's Framework — Keep This

A reusable checklist for any new dataset you're handed, in any future job:

1. **Understand the business problem** — *What decision does this support?*
2. **Ask analytical questions** — *What specific, measurable question breaks this down?*
3. **Identify required data** — *What data would actually answer this?*
4. **Gather data** — *Where does it live, and in what format?*
5. **Understand the grain** — *What does one row represent?*
6. **Assess data quality** — *What's actually in here, before I touch anything?*
7. **Clean** — *Why is this wrong/missing, and what's the right fix?*
8. **Wrangle** — *Is this in the shape I need?*
9. **Engineer features** — *What derived variable would capture the business concept?*
10. **Explore** — *What does the data show, one relationship at a time?*
11. **Calculate descriptive statistics** — *What does this number actually mean here?*
12. **Identify patterns** — *What's consistent, and what's surprising?*
13. **Validate findings** — *Is this real, or a data-quality artifact? Is the sample size reliable?*
14. **Draw evidence-based conclusions** — *What can I actually claim?*
15. **Communicate business implications + limitations** — *What should someone DO with this, and what should they NOT conclude?*

See the companion DOCX workbook (Section 19) for the fully-illustrated one-page version of this framework, designed to be kept on your desk.
